In [0]:
from pyspark.sql.functions import count, col
from datetime import date
from dateutil.relativedelta import relativedelta

four_months_ago_start = date.today() - relativedelta(months=4)



df_trips = spark.read.table("nyctaxi.02_silver.yellow_trips_cleansed").filter(col("tpep_pickup_datetime") > four_months_ago_start)
                                                                              
df_zones = spark.read.table("nyctaxi.02_silver.taxi_zone_lookup")

df_pickup = df_trips.join(df_zones, 
              df_trips.pu_location_id == df_zones.location_id, 
              "left").select(
                  df_trips.vendor,
                  df_trips.tpep_pickup_datetime,
                  df_trips.tpep_dropoff_datetime,
                  df_trips.trip_duration_minutes,
                  df_trips.passenger_count,
                  df_trips.trip_distance,
                  df_trips.rate_type,
                  df_trips.store_and_fwd_flag,
                  df_zones.borough.alias("pickup_borough"),
                  df_zones.zone.alias("pickup_zone"),
                  df_trips.do_location_id,
                  df_trips.payment_type,
                  df_trips.fare_amount,
                  df_trips.extra,
                  df_trips.mta_tax,
                  df_trips.tip_amount,
                  df_trips.tolls_amount,
                  df_trips.improvement_surcharge,
                  df_trips.total_amount,
                  df_trips.congestion_surcharge,
                  df_trips.airport_fee,
                  df_trips.cbd_congestion_fee,
                  df_trips.processed_timestamp,
              )

df_full = df_pickup.join(df_zones,
                         df_pickup.do_location_id == df_zones.location_id,
                         "left").select(
                            df_pickup.vendor,
                            df_pickup.tpep_pickup_datetime,
                            df_pickup.tpep_dropoff_datetime,
                            df_pickup.trip_duration_minutes,
                            df_pickup.passenger_count,
                            df_pickup.trip_distance,
                            df_pickup.rate_type,
                            df_pickup.store_and_fwd_flag,
                            df_pickup.pickup_borough,
                            df_zones.borough.alias("dropoff_borough"),
                            df_pickup.pickup_zone,
                            df_zones.zone.alias("dropoff_zone"),
                            df_pickup.payment_type,
                            df_pickup.fare_amount,
                            df_pickup.extra,
                            df_pickup.mta_tax,
                            df_pickup.tip_amount,
                            df_pickup.tolls_amount,
                            df_pickup.improvement_surcharge,
                            df_pickup.total_amount,
                            df_pickup.congestion_surcharge,
                            df_pickup.airport_fee,
                            df_pickup.cbd_congestion_fee,
                            df_pickup.processed_timestamp
                         )
                  
df_full.display()

In [0]:
df_full.write.mode("append").saveAsTable("nyctaxi.02_silver.yellow_trips_enriched")

In [0]:
spark.read.table("nyctaxi.02_silver.yellow_trips_enriched").display()